## LEVEL 4
1. Add Field to record res10 id and res9 id

#### Initialize

In [2]:
import sys
from pathlib import Path
import pandas as pd

HERE = Path.cwd()
PARENT = HERE.parent.parent.parent  # server/scripts
if str(PARENT) not in sys.path:
    sys.path.insert(0, str(PARENT))

LEVEL3_BUCKET_PATH = PARENT / "server/out/places_level3"
LEVEL4_BUCKET_PATH = PARENT / "server/out/places_level4"
LEVEL4_BUCKET_PATH.mkdir(parents=True, exist_ok=True)
LEVEL3_BUCKET = [f for f in LEVEL3_BUCKET_PATH.rglob("*.csv") if f.is_file() and not f.stem.startswith("timetable")]
DF_LEVEL3 = pd.concat([pd.read_csv(f) for f in LEVEL3_BUCKET], ignore_index=True)

#### Add H3 R9 + R10

In [3]:
from server.scripts.h3.h3_api import _h3_latlng_to_cell

df_level4 = DF_LEVEL3.copy()

# Add H3 Res9 Res10
df_level4["h3_res9"] = df_level4.apply(lambda r: _h3_latlng_to_cell(float(r['lat']), float(r['lon']), 9), axis=1)
df_level4["h3_res10"] = df_level4.apply(lambda r: _h3_latlng_to_cell(float(r['lat']), float(r['lon']), 10), axis=1)

#### Build Region Grid

In [4]:
from server.scripts.h3.h3_api import _h3_get_neighbours

df_level4['local_tile'] = df_level4['h3_res9']
grid_res9 = pd.DataFrame({
    'local_tile': df_level4['local_tile'].unique(),
}).sort_values('local_tile').reset_index(drop=True)

grid_res9['local_tiles'] = grid_res9['local_tile'].apply(
    lambda tile: _h3_get_neighbours(tile, k=2) | {tile}  # include self
)

if not (Path("grid_res9.csv")).exists():
    for row in grid_res9.itertuples():
        places = df_level4[df_level4['local_tile'] == row.local_tile]
        grid_res9.at[row.Index, 'density'] = len(places)
        local_places = df_level4[df_level4['local_tile'].isin(row.local_tiles)]
        grid_res9.at[row.Index, 'local_density'] = len(local_places)

    grid_res9.to_csv("grid_res9.csv", index=False)

#### Local Representation

In [5]:
from server.scripts.clean_places_level_4.get_local import get_local_representation_ratio

rep_ratio_per_cell = []
for _, region_row in grid_res9.iterrows():
    local_composition = get_local_representation_ratio(df_level4, region_row['local_tiles'], region_row['local_tile'])
    if not local_composition.empty:
        rep_ratio_per_cell.append(local_composition)
local_representation = pd.concat(rep_ratio_per_cell, ignore_index=True)
# local_representation_ratio.to_csv("grid_res9_compositions.csv", index=False)

In [6]:
# Map local representation ratio and count onto each place
local_scores = local_representation[["local_tile", "cuisineType", 
    "venueType", "local_rep_ratio", "local_rep_count", "local_total"]]
    
df_parsed_local = df_level4.merge( local_scores, on=["local_tile", "cuisineType", "venueType"], how="left")

#### Global Score

In [7]:
df_global_comp = df_level4.loc[df_level4["cuisineType"].ne("Unspecified")].copy()
global_composition = (df_global_comp.groupby(["cuisineType", "venueType"], dropna=False)
    .size().rename("global_count").reset_index() # n_global
)
global_composition["global_ratio"] = global_composition["global_count"] / max(len(df_global_comp), 1) #p_global
global_composition.drop(columns=["global_count"], inplace=True)

In [8]:
# Map global representation ratio and count onto each place    
df_parsed_global = df_parsed_local.merge( global_composition, on=["cuisineType", "venueType"], how="left")
df_parsed_global['rep_delta'] = df_parsed_global['local_rep_ratio'] - df_parsed_global['global_ratio']
df_parsed_global.drop(columns=["global_ratio"], inplace=True)

#### Export

In [9]:
df_parsed_global.to_csv(LEVEL4_BUCKET_PATH / "places_level4.csv", index=False)